<a href="https://colab.research.google.com/github/adityab-tech/PRISMx/blob/main/notebooks/02_Fine_Tuning_Kronos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PRISM - Deliverable 2--Fine-Tuning Kronos on Indian Equity Data

### Objectives

- Load Kronos Foundation Model
- Load Tokenizer
- Load Processed Dataset
- Fine-tune using LoRA
- Evaluate against Zero-Shot
- Save Best Model



#Setup

In [ ]:
!git clone https://github.com/shiyu-coder/Kronos.git

fatal: destination path 'Kronos' already exists and is not an empty directory.


In [ ]:
!git clone https://github.com/NeoQuasar/Kronos.git

fatal: destination path 'Kronos' already exists and is not an empty directory.


In [ ]:
!pip install -q -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import random
import sys
import warnings
import yaml
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from train_sequential import SequentialTrainer

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

warnings.filterwarnings("ignore")

In [ ]:
!pip -q install transformers peft bitsandbytes accelerate huggingface_hub sentencepiece pyyaml safetensors

In [ ]:
sys.path.append("/content/Kronos")
sys.path.append("/content/Kronos/finetune_csv")

In [ ]:
PROJECT_ROOT = "/content/drive/MyDrive/PRISM"

PROCESSED_PATH = os.path.join(PROJECT_ROOT,"data/processed")
MODEL_PATH = os.path.join(PROJECT_ROOT,"models")

In [ ]:
from model import Kronos, KronosTokenizer
from train_sequential import SequentialTrainer
from config_loader import CustomFinetuneConfig
from finetune_base_model import create_dataloaders

In [ ]:
dataset_path = os.path.join(PROCESSED_PATH,"RELIANCE.NS.csv"
)
df = pd.read_csv(dataset_path)
df.head()

,Date,Close,High,Low,Open,Volume,Return,Log_Return,MA5,MA10,MA20,EMA20,Volatility,Volume_Change,Volume_MA20,Beta
0,2020-03-27,474.505524,493.074299,465.866824,487.597187,41657940,-0.000563,-0.000563,448.990179,443.929416,496.833708,495.441343,0.069612,-0.089334,47997996.65,1.236910
1,2020-03-30,458.853424,478.602224,454.200113,463.373180,30229866,-0.032986,-0.033542,462.028387,444.586221,490.194370,491.956779,0.069439,-0.274331,47543701.50,1.221455
2,2020-03-31,495.946472,503.093419,466.668378,478.223761,44292810,0.080839,0.077737,477.199561,449.295203,485.687991,492.336750,0.072373,0.465200,48283317.25,1.247226
3,2020-04-01,481.118164,500.777879,465.421544,499.731437,41597459,-0.029899,-0.030355,477.039258,454.280273,479.845731,491.268313,0.072290,-0.060853,48993250.30,1.235940
4,2020-04-03,479.782257,505.164047,470.364285,505.164047,41367807,-0.002777,-0.002781,478.041168,461.393848,474.006810,490.174403,0.072288,-0.005521,49956377.60,1.229398


In [ ]:
df = df.rename(
    columns={"Date":"timestamps","Open":"open","High":"high","Low":"low","Close":"close","Volume":"volume"
    })
df["amount"] = 0
df["timestamps"] = pd.to_datetime(df["timestamps"])
df.head()

,timestamps,close,high,low,open,volume,Return,Log_Return,MA5,MA10,MA20,EMA20,Volatility,Volume_Change,Volume_MA20,Beta,amount
0,2020-03-27,474.505524,493.074299,465.866824,487.597187,41657940,-0.000563,-0.000563,448.990179,443.929416,496.833708,495.441343,0.069612,-0.089334,47997996.65,1.236910,0
1,2020-03-30,458.853424,478.602224,454.200113,463.373180,30229866,-0.032986,-0.033542,462.028387,444.586221,490.194370,491.956779,0.069439,-0.274331,47543701.50,1.221455,0
2,2020-03-31,495.946472,503.093419,466.668378,478.223761,44292810,0.080839,0.077737,477.199561,449.295203,485.687991,492.336750,0.072373,0.465200,48283317.25,1.247226,0
3,2020-04-01,481.118164,500.777879,465.421544,499.731437,41597459,-0.029899,-0.030355,477.039258,454.280273,479.845731,491.268313,0.072290,-0.060853,48993250.30,1.235940,0
4,2020-04-03,479.782257,505.164047,470.364285,505.164047,41367807,-0.002777,-0.002781,478.041168,461.393848,474.006810,490.174403,0.072288,-0.005521,49956377.60,1.229398,0


In [ ]:
save_path = "/content/Kronos/finetune_csv/data/prism_train.csv"
df.to_csv(save_path,index=False)
print(save_path)

/content/Kronos/finetune_csv/data/prism_train.csv


In [ ]:
config = {

"data":{
"data_path":save_path,
"lookback_window":30,
"predict_window":5,
"max_context":30,
"clip":5,
"train_ratio":0.8,
"val_ratio":0.2,
"test_ratio":0.0
},

"training":{
"tokenizer_epochs":2,
"basemodel_epochs":2,
"batch_size":32,
"learning_rate":1e-4,
"num_workers":2
},

"model_paths":{
"exp_name":"PRISM",
"pretrained_tokenizer":"NeoQuasar/Kronos-Tokenizer-base",
"pretrained_predictor":"NeoQuasar/Kronos-base",
"base_save_path":"/content/Kronos/checkpoints"
}
}

In [ ]:
config_path = "/content/Kronos/finetune_csv/configs/prism.yaml"
with open(config_path,"w") as f:
    yaml.dump(config,f)
cfg = CustomFinetuneConfig(
    config_path)
cfg.print_config_summary()

Kronos finetuning configuration summary
Experiment name: PRISM
Data path: /content/Kronos/finetune_csv/data/prism_train.csv
Lookback window: 30
Predict window: 5
Tokenizer training epochs: 2
Basemodel training epochs: 2
Batch size: 32
Tokenizer learning rate: 0.0002
Predictor learning rate: 4e-05
Train tokenizer: True
Train basemodel: True
Skip existing: False
Use pre-trained tokenizer: True
Use pre-trained predictor: True
Base save path: /content/Kronos/checkpoints
Tokenizer save path: /content/Kronos/checkpoints/tokenizer
Basemodel save path: /content/Kronos/checkpoints/basemodel


In [ ]:
train_loader, val_loader, *_ = create_dataloaders(cfg)

Creating data loaders...
Original data time range: 2020-03-27 00:00:00 to 2024-12-31 00:00:00
Original data total length: 1118 records
[TRAIN] Training set: first 894 time points (0.8)
[TRAIN] Training set time range: 2020-03-27 00:00:00 to 2024-02-01 00:00:00
[TRAIN] Data length after split: 894 records
[TRAIN] Data length: 894, Available samples: 859
Original data time range: 2020-03-27 00:00:00 to 2024-12-31 00:00:00
Original data total length: 1118 records
[VAL] Validation set: time points 895 to 1118 (0.2)
[VAL] Validation set time range: 2024-02-02 00:00:00 to 2024-12-31 00:00:00
[VAL] Data length after split: 224 records
[VAL] Data length: 224, Available samples: 189
Training set size: 859, Validation set size: 189


In [ ]:
batch_x, batch_stamp = next(iter(train_loader))

In [ ]:
print(batch_x.shape)
print(batch_stamp.shape)

torch.Size([32, 36, 6])
torch.Size([32, 36, 5])


In [ ]:
tokenizer = KronosTokenizer.from_pretrained("NeoQuasar/Kronos-Tokenizer-base")
model = Kronos.from_pretrained("NeoQuasar/Kronos-base")

config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/15.8M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/409M [00:00<?, ?B/s]

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
batch_x = batch_x.to(device)

In [ ]:
with torch.no_grad():
    (
        (reconstruction, encoder_output),
        bsq_loss,
        quantized,
        token_ids
    ) = tokenizer(batch_x)

In [ ]:
print(reconstruction.shape)
print(encoder_output.shape)
print(quantized.shape)
print(token_ids.shape)
print(bsq_loss)

torch.Size([32, 36, 6])
torch.Size([32, 36, 6])
torch.Size([32, 36, 20])
torch.Size([32, 36])
tensor(-0.0674)


In [ ]:
print(token_ids.min())
print(token_ids.max())
print(token_ids.unique().numel())

tensor(13741)
tensor(1024423)
338


In [ ]:
print(token_ids[:2])

tensor([[  30645,   30661,   32733,   30685,   30669,  278479,  691912,  998363,
          278415,  474079,  278415,  471979,  276399,  278415,  474031,  471979,
          459179, 1018179,  473995,  471983,  985483, 1018115,  888146, 1019219,
          675416,  998363, 1018115, 1016227,  887075,  887106,  886016,  839272,
          885828,  885792,  888146,  885074],
        [1019339,  887123,  885010, 1018251,  884994, 1018115,  998299, 1018131,
          887042,  885760,  885798, 1018243, 1018179,  887874,  885026,  884738,
         1018131,  677582,  669268,   30621,  816863,  474079,  278415,  461259,
         1018275,  278415,   30605,   30613,   30645,   16269,  278447,   30605,
           30597,  474027,  278415,   30637]])


In [ ]:
decoded = tokenizer.decode(token_ids)
print(decoded.shape)

torch.Size([32, 36, 6])


In [ ]:
mse = (( batch_x.cpu()-decoded.cpu())**2).mean()
print(mse.item())

0.036786999553442


In [ ]:
trainer = SequentialTrainer(config_path)
print(trainer)

Using device: cpu (rank=0, world_size=1, local_rank=0)
Kronos finetuning configuration summary
Experiment name: PRISM
Data path: /content/Kronos/finetune_csv/data/prism_train.csv
Lookback window: 30
Predict window: 5
Tokenizer training epochs: 2
Basemodel training epochs: 2
Batch size: 32
Tokenizer learning rate: 0.0002
Predictor learning rate: 4e-05
Train tokenizer: True
Train basemodel: True
Skip existing: False
Use pre-trained tokenizer: True
Use pre-trained predictor: True
Base save path: /content/Kronos/checkpoints
Tokenizer save path: /content/Kronos/checkpoints/tokenizer
Basemodel save path: /content/Kronos/checkpoints/basemodel


In [ ]:
print(dir(trainer))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_check_existing_models', '_create_directories', '_setup_device', '_setup_distributed', 'config', 'device', 'local_rank', 'rank', 'run_training', 'train_basemodel_phase', 'train_tokenizer_phase', 'world_size']
